<a href="https://colab.research.google.com/github/anthonyg2311/multivariate-regression-analysis/blob/main/Copy_of_Actual_(Batch)_Multi_Logistic_Reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
raw_data = pd.read_csv('titanic_train.csv')
raw_data = raw_data.drop(raw_data.columns[0], axis = 1)
raw_data = raw_data.drop(raw_data.columns[2], axis = 1)
raw_data = raw_data.drop(raw_data.columns[6], axis = 1)
raw_data = raw_data.drop(raw_data.columns[7], axis = 1)

In [ ]:
# Classifying string values as numerical representations

#Sex
# male = 1; female = 0
for i in range(len(raw_data['Sex'])):
  if(raw_data['Sex'].iloc[i] == 'male'):
    raw_data['Sex'].iloc[i] = 1
  elif(raw_data['Sex'].iloc[i] == 'female'):
    raw_data['Sex'].iloc[i] = 0

#Embarked
# S = 0, C = 1, Q = 2
for i in range(len(raw_data['Embarked'])):
  if(raw_data['Embarked'].iloc[i] == 'S'):
    raw_data['Embarked'].iloc[i] = 0
  elif(raw_data['Embarked'].iloc[i] == 'C'):
    raw_data['Embarked'].iloc[i] =1
  elif(raw_data['Embarked'].iloc[i] == 'Q'):
    raw_data['Embarked'].iloc[i] = 2

In [4]:
#Function that fills in NULL entries with the average of each respective column's filled entries.
def fill(col):
  mean = 0
  count =0
  for i in range(len(col)):
    if(not pd.isna(col.iloc[i])):
      mean += col.iloc[i]
      count += 1
  mean = round(mean / count)
  for i in range(len(col)):
    if(pd.isna(col.iloc[i])):
      col.iloc[i] = mean
  return col

In [ ]:
for col in raw_data.columns:
  raw_data[col] = fill(raw_data[col])
data = raw_data

In [ ]:
#Normalization function to ensure each weight is treated equally.
def normalize(data):
  for j in range(data.shape[1]):
    for i in range(len(data)):
      data.iloc[i, j] = (data.iloc[i, j] - min(data.iloc[:, j])) / (max(data.iloc[:, j]) - min(data.iloc[:, j]))

  return data
data = normalize(data)

In [7]:
X = data.drop('Survived', axis = 1)
Y = data['Survived']
X = np.array(X)
Y = np.array(Y)

In [8]:
#Cost function for data using
def cost(thetaVec, b, numExamples):
  cost =0
  for i in range(numExamples):
    cost+= (-1/numExamples) * Y[i] * np.log(sigmoid(np.dot(thetaVec, X[i])+b)) + (1 - Y[i]) * np.log(1 - (sigmoid(np.dot(thetaVec, X[i])+b)))
  return -cost/numExamples

In [9]:
def sigmoid(t):
  return 1/(1+np.exp(-t))

In [10]:
def gD(lr, thetaVec, b, numExamples):

  #Initialization
  gradientVec = np.zeros(len(X[0]))
  db = 0

  #Gradient Calculation
  for j in range(len(gradientVec)):
    for i in range(numExamples):
      gradientVec[j] += (1/numExamples) * ((sigmoid(np.dot(thetaVec, X[i])+b)) - Y[i]) * X[i][j]
      db+= (1/numExamples) * ((sigmoid(np.dot(thetaVec, X[i])+b)) - Y[i])

  #One step of Gradient Descent
  for i in range(len(thetaVec)):
    thetaVec[i] = thetaVec[i] - (lr * gradientVec[i])
  b = b - (lr * db)
  return thetaVec, b

#Initializing parameters for Gradient Descent
lr = 0.1
thetaVec = np.zeros(len(X[0]))
b = 0
numExamples = len(data)
epochs = 1000 #Number of iterations

#Running
for i in range(epochs):
  thetaVec, b = gD(lr, thetaVec, b, numExamples)

#Outputs
print(thetaVec, b) #Vector which contains my weights for Logistic Regression
print(cost(thetaVec, b, numExamples)) #Error

[-1.90790978 -2.39350676 -0.49438001 -0.31551203 -0.109024    0.52816675
  0.45273757] 2.506156568415264
0.20137746403841275


In [11]:
#Sklearn model comparison
from sklearn.linear_model import LogisticRegression
lR = LogisticRegression()
lR.fit(X, Y)
print(lR.coef_, lR.intercept_)
print(lR.score(X, Y))
coef_array = np.array(lR.coef_).flatten()  # Flatten to a 1D array
intercept_array = np.array(lR.intercept_)
print(cost(coef_array, intercept_array, numExamples))

[[-2.84643731 -2.51493786 -2.01226165 -0.97711688 -0.51467502  0.80647396
   0.50495637]] [3.92305148]
0.8002244668911336
[0.19448351]


In [19]:
print("Comparison:\n", "My model: ",cost(thetaVec, b, numExamples), "\n","Sklearn Model: ",cost(coef_array, intercept_array, numExamples))

Comparison:
 My model:  0.20137746403841275 
 Sklearn Model:  [0.19448351]
